# Step 15.5 — Full Forward Shape Trace：从 `(B,T)` 一路追到 `(B,T,V)`

Step 15.4 已经真正实现了 `MiniGPT(nn.Module)`。

这一节不再增加新模块，而是做三件收束工作：

1. **逐行追踪完整 forward shape**，包括每个 Transformer Block 内部；
2. **严格计算当前 MiniGPT 的参数数量**；
3. **把每一段代码重新连接到前面 MLP → Seq2Seq → Attention → Transformer → GPT 的历史目的。**

本节仍然只准备 Notebook 与 trace / count 教学代码。

> **没有执行 Notebook、forward、assertion、parameter-count code、测试或实验。**

本轮完成即停，不进入 Step 16。


## 1. 先固定当前 MiniGPT 的 Architecture Convention

沿用 Step 15.4，不修改任何已经审核通过的设计：

```text
V       = vocabulary size
C       = model width
H       = number of heads
D       = C / H
N       = number of Transformer Blocks
T_max   = learned absolute-position table length
T       = current sequence length, 1 <= T <= T_max
```

Trainable design：

```text
token embedding             (V,C)
position embedding          (T_max,C)

per Block:
  RMSNorm 1                 C
  fused QKV                 (3C,C), bias=False
  c_proj                    (C,C),  bias=False
  RMSNorm 2                 C
  MLP fc1                   (4C,C) + bias (4C)
  MLP fc2                   (C,4C) + bias (C)

final RMSNorm               C
LM head                     (V,C), bias=False
```

Token embedding 与 LM head **不做 weight tying**。

Causal mask、position IDs、`head_dim`、scale、当前 T 都不是 trainable parameters。


## 2. 顶层 Forward Shape：先看整条主干

输入 token IDs：

$$
token\_ids:(B,T).
$$

### Token embedding

$$
(B,T)
\rightarrow
(B,T,C).
$$

### Position embedding

$$
position\_ids:(T)
\rightarrow
(T,C).
$$

broadcast addition：

$$
(B,T,C)+(T,C)
\rightarrow
\boxed{(B,T,C)}.
$$

### Block stack

每个 Block：

$$
(B,T,C)\rightarrow(B,T,C).
$$

因此 N 层后仍然：

$$
(B,T,C).
$$

### Final RMSNorm

$$
(B,T,C)\rightarrow(B,T,C).
$$

### LM head

$$
\boxed{
(B,T,C)
\rightarrow
(B,T,V)
}.
$$

所以完整主线：

$$
\boxed{
(B,T)
\rightarrow
(B,T,C)
\rightarrow
\cdots
\rightarrow
(B,T,C)
\rightarrow
(B,T,V)
}.
$$


## 3. 一个 Block 内部的完整 Shape Trace

设进入某个 Block：

$$
x:(B,T,C),
\qquad
C=H\times D.
$$

### Attention branch

```text
x                                  (B,T,C)
↓ norm1
normalized                         (B,T,C)
↓ fused qkv_proj C→3C
qkv                                (B,T,3C)
↓ split
q, k, v                            each (B,T,C)
↓ reshape C=H×D
q, k, v                            each (B,T,H,D)
↓ swap T/H
q, k, v                            each (B,H,T,D)
↓ q @ k^T
scores                             (B,H,T,T)
↓ + causal mask (T,T)
masked scores                      (B,H,T,T)
↓ softmax over Key axis
weights                            (B,H,T,T)
↓ weights @ v
head outputs                       (B,H,T,D)
↓ swap H/T back
token-major heads                  (B,T,H,D)
↓ merge H×D=C
merged                             (B,T,C)
↓ c_proj C→C
attention update                   (B,T,C)
↓ residual add
x1                                 (B,T,C)
```

### MLP branch

```text
x1                                 (B,T,C)
↓ norm2
normalized                         (B,T,C)
↓ fc1 C→4C
hidden                             (B,T,4C)
↓ GELU
activated                          (B,T,4C)
↓ fc2 4C→C
mlp update                         (B,T,C)
↓ residual add
block output                       (B,T,C)
```

因此 Block 内部会暂时产生 `3C`、`H×D`、`T×T`、`4C` 等 shapes，但 Block 边界始终恢复到 `(B,T,C)`。


## 4. 每段 Forward Code 的“历史目的”是什么？

| 当前 MiniGPT 部分 | 历史 / 技术问题 | 它解决什么 |
|---|---|---|
| Token Embedding | 离散 token ID 不能直接当连续语义向量 | `ID -> C-dimensional learned representation` |
| Position Embedding | Transformer 去掉 recurrence 后没有 RNN 那种隐式顺序传播 | 给 position-specific signal |
| Causal mask | Autoregressive prediction 禁止 future leakage | 强制 `j<=i` |
| Self-Attention | RNN 长路径 / 串行 dependency；Attention 的直接检索思想 | 让 prefix positions 直接交互 |
| Multi-Head split | 一个 projection view 表示受限 | 多个 learned interaction subspaces |
| Residual | 深层更新与 gradient path | `x + correction`，提供 identity route |
| RMSNorm | 深层 residual-stream scale 管理 | 不做 mean-centering 的 re-scaling |
| Position-wise MLP | Attention 主要负责跨位置通信 | 每个 token 内做 nonlinear feature transformation |
| Block × N | 单层 transformation 深度有限 | 重复 refinement，且各层参数独立 |
| Final RMSNorm | 当前 Pre-Norm stack 的最终 residual stream 未自动 exit-normalize | 在 LM head 前规范最终 features |
| LM Head | hidden width C 不是 vocabulary prediction space | `C -> V` next-token logits |

这样可以看到，我们最终的十几行顶层代码其实是前面整条历史链逐步留下来的结果，而不是一开始凭空规定出来的架构。


## 5. 顶层 Parameter Count：Embedding / Final Norm / LM Head

先计算 Block 之外。

### Token embedding

$$
E_{tok}:(V,C)
$$

所以：

$$
\boxed{VC}.
$$

### Learned absolute position embedding

$$
E_{pos}:(T_{max},C)
$$

参数：

$$
\boxed{T_{max}C}.
$$

注意：即使某个 batch 当前只有 `T<T_max`，整张 learned position table 仍然都是模型参数。

### Final RMSNorm

只有 learned gamma：

$$
\boxed{C}.
$$

### LM head

当前不 weight-tie、bias=False：

MLX stored weight：

$$
(V,C)
$$

参数：

$$
\boxed{VC}.
$$

所以 Block 外部总计：

$$
\boxed{
2VC+T_{max}C+C
}.
$$


## 6. 每个 Block 的 Parameter Count 再核一次

### Attention

fused QKV：

$$
(3C,C)
\Rightarrow
3C^2.
$$

`c_proj`：

$$
(C,C)
\Rightarrow
C^2.
$$

所以：

$$
\boxed{4C^2}.
$$

### 两套 RMSNorm

$$
C+C
=
\boxed{2C}.
$$

### MLP

`fc1`：

$$
4C^2+4C.
$$

`fc2`：

$$
4C^2+C.
$$

合计：

$$
\boxed{8C^2+5C}.
$$

所以一个 Block：

$$
\boxed{
4C^2+2C+8C^2+5C
=
12C^2+7C
}.
$$

N 个独立 Blocks：

$$
\boxed{
N(12C^2+7C)
}.
$$


## 7. 当前 MiniGPT 的总参数公式

把所有部分相加：

$$
P
=
VC
+
T_{max}C
+
N(12C^2+7C)
+
C
+
VC.
$$

因此：

$$
\boxed{
P
=
2VC
+
T_{max}C
+
N(12C^2+7C)
+
C
}.
$$

也可以整理成：

$$
\boxed{
P
=
12NC^2
+
C(2V+T_{max}+7N+1)
}.
$$

### 为什么公式里没有 H？

在当前 full-width QKV implementation 中，head split 只是把 C 重新组织为：

$$
C=H\times D.
$$

QKV / output projection 的总 width 仍然由 C 决定。

所以在固定 C 下，改变 H 不改变这些 projection matrices 的参数总数；它改变的是 tensor organization / head dimension D。

前提仍然是：

$$
C\bmod H=0.
$$


## 8. 用 Step 15.4 的 Tiny Config 手算一次

沿用：

```text
V      = 32
C      = 16
H      = 4
N      = 2
T_max  = 16
```

### Token embedding

$$
32\times16
=
512.
$$

### Position embedding

$$
16\times16
=
256.
$$

### One Block

$$
12(16^2)+7(16)
=
3072+112
=
3184.
$$

两个 Blocks：

$$
2\times3184
=
6368.
$$

### Final RMSNorm

$$
16.
$$

### LM head

$$
32\times16
=
512.
$$

总计：

$$
512+256+6368+16+512
=
\boxed{7664}.
$$

因为没有 weight tying，所以两个 `VC=512` 必须分别计入。


In [ ]:
import math


def expected_minigpt_parameter_count(
    vocab_size: int,
    dims: int,
    num_blocks: int,
    max_seq_len: int,
) -> int:
    token_embedding = vocab_size * dims
    position_embedding = max_seq_len * dims
    one_block = 12 * dims * dims + 7 * dims
    block_stack = num_blocks * one_block
    final_norm = dims
    lm_head = vocab_size * dims

    return (
        token_embedding
        + position_embedding
        + block_stack
        + final_norm
        + lm_head
    )


expected_for_tiny_config = expected_minigpt_parameter_count(
    vocab_size=32,
    dims=16,
    num_blocks=2,
    max_seq_len=16,
)

print('expected parameter count:', expected_for_tiny_config)

# Hand-derived expected value: 7664.
# This cell has NOT been executed in the master notebook.


## 9. 如何在以后执行时，用 MLX Parameter Tree 对照手算？

MLX `model.trainable_parameters()` 返回递归 parameter tree。

`mlx.utils.tree_flatten` 可以把 tree 展平成：

```text
(parameter path, mx.array)
```

pairs。

于是可以对所有 array shapes 做：

$$
\operatorname{numel}(p)
=
\prod_i shape_i.
$$

再求和。

重要边界：

- causal mask 是 forward-local，不应该出现在这棵 trainable tree；
- position IDs 也是 forward-local；
- Python metadata（T_max、head_dim、scale）不计入；
- token embedding 与 LM head 当前独立，因此都计入。

下面只准备 parameter-tree count code，本轮不运行。


In [ ]:
from mlx.utils import tree_flatten


def numel(array):
    return math.prod(array.shape)


def count_trainable_parameters(model):
    flat = tree_flatten(model.trainable_parameters())
    return sum(numel(array) for _, array in flat)


# PREPARED CHECK ONLY — do not execute in the master.
# model = MiniGPT(
#     vocab_size=32,
#     dims=16,
#     num_heads=4,
#     num_blocks=2,
#     max_seq_len=16,
# )
# actual = count_trainable_parameters(model)
# expected = expected_minigpt_parameter_count(32, 16, 2, 16)
# assert actual == expected == 7664

# No model construction, parameter traversal, or assertion has run here.


## 10. 为 Work Copy 准备一个真正逐步打印 Shape 的 Trace Helper

普通：

```python
logits = model(token_ids)
```

只能看到最终结果。

为了教学，可以写一个 `trace_forward_shapes` helper，复现同一个 forward 的中间步骤，并打印每个 tensor shape。

这个 helper 不是新的模型；它只是把 Step 15.4 已有 module internals 展开观察。

特别是每个 Block 都会打印：

```text
norm1
qkv
q/k/v before heads
q/k/v after heads
scores
weights
attention output
residual 1
norm2
MLP hidden
MLP output
residual 2
```

本轮只写 helper，不执行。


In [ ]:
def trace_forward_shapes(model, token_ids):
    if len(token_ids.shape) != 2:
        raise ValueError('token_ids must have shape (B,T)')

    B, T = token_ids.shape
    if T < 1 or T > model.max_seq_len:
        raise ValueError('invalid sequence length for this model')

    print('token_ids                 ', token_ids.shape)

    position_ids = mx.arange(T)
    print('position_ids              ', position_ids.shape)

    token_features = model.token_embedding(token_ids)
    print('token_features            ', token_features.shape)

    position_features = model.position_embedding(position_ids)
    print('position_features         ', position_features.shape)

    x = token_features + position_features
    print('initial hidden            ', x.shape)

    for block_index, block in enumerate(model.blocks):
        print(f'--- block {block_index} ---')

        n1 = block.norm1(x)
        print('norm1                     ', n1.shape)

        attn = block.attn
        Bx, Tx, Cx = n1.shape
        H = attn.num_heads
        D = attn.head_dim

        qkv = attn.qkv_proj(n1)
        print('qkv                       ', qkv.shape)

        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)
        print('q/k/v split               ', q.shape, k.shape, v.shape)

        q = mx.reshape(q, (Bx, Tx, H, D))
        k = mx.reshape(k, (Bx, Tx, H, D))
        v = mx.reshape(v, (Bx, Tx, H, D))
        print('q/k/v BTHD                ', q.shape, k.shape, v.shape)

        q = mx.swapaxes(q, 1, 2)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)
        print('q/k/v BHTD                ', q.shape, k.shape, v.shape)

        scores = mx.matmul(q, mx.swapaxes(k, -1, -2)) * attn.scale
        print('attention scores          ', scores.shape)

        future = mx.triu(mx.ones((Tx, Tx)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )
        print('causal mask               ', mask.shape)

        weights = mx.softmax(scores + mask, axis=-1)
        print('attention weights         ', weights.shape)

        head_out = mx.matmul(weights, v)
        print('head outputs              ', head_out.shape)

        token_major = mx.swapaxes(head_out, 1, 2)
        print('token-major heads         ', token_major.shape)

        merged = mx.reshape(token_major, (Bx, Tx, Cx))
        print('merged heads              ', merged.shape)

        attn_update = attn.c_proj(merged)
        print('attention update          ', attn_update.shape)

        x = x + attn_update
        print('after attention residual  ', x.shape)

        n2 = block.norm2(x)
        print('norm2                     ', n2.shape)

        mlp_hidden = block.mlp.fc1(n2)
        print('MLP fc1                   ', mlp_hidden.shape)

        mlp_activated = block.mlp.act(mlp_hidden)
        print('MLP activation            ', mlp_activated.shape)

        mlp_update = block.mlp.fc2(mlp_activated)
        print('MLP fc2                   ', mlp_update.shape)

        x = x + mlp_update
        print('after MLP residual        ', x.shape)

    x = model.final_norm(x)
    print('final norm                ', x.shape)

    logits = model.lm_head(x)
    print('logits                    ', logits.shape)

    return logits


# PREPARED USAGE — intentionally commented / unexecuted:
# model = MiniGPT(32, 16, 4, 2, 16)
# sample_ids = mx.array([[1, 7, 3, 9, 2]])
# traced_logits = trace_forward_shapes(model, sample_ids)
# assert traced_logits.shape == (1, 5, 32)

# No forward trace has been executed in this master notebook.


## 11. Tiny Config 的 Expected Full Trace

继续使用：

```text
B = 1
T = 5
V = 32
C = 16
H = 4
D = 4
N = 2
T_max = 16
```

那么一个 Block 内：

```text
input                      (1,5,16)
norm1                      (1,5,16)
qkv                        (1,5,48)
q/k/v                      each (1,5,16)
q/k/v BTHD                 each (1,5,4,4)
q/k/v BHTD                 each (1,4,5,4)
scores                     (1,4,5,5)
causal mask                (5,5)
weights                    (1,4,5,5)
head outputs               (1,4,5,4)
token-major                (1,5,4,4)
merged                     (1,5,16)
attention update           (1,5,16)
after residual             (1,5,16)
norm2                      (1,5,16)
MLP fc1                    (1,5,64)
GELU                       (1,5,64)
MLP fc2                    (1,5,16)
Block output               (1,5,16)
```

两个 Blocks 后仍为：

$$
(1,5,16).
$$

Final RMSNorm：

$$
(1,5,16).
$$

LM head：

$$
\boxed{(1,5,32)}.
$$

这些都是从代码 contract 静态推出的 expected shapes，不是本轮实际运行输出。


## 12. 从最早课程主线看，MiniGPT 为什么最终长成这样？

```text
MLP
  -> 学会 feature transformation

RNN / LSTM / GRU
  -> 学会 sequence state 与 autoregressive dependency
  -> 同时暴露 time-serial / long-path limitations

Seq2Seq
  -> 把 source understanding 与 target generation 拆成 encoder / decoder roles

Attention
  -> decoder 不必只依赖 fixed final state，可直接检索 source states

Self-Attention
  -> 同一 sequence positions 彼此直接交互

Causal Self-Attention
  -> 把 direct interaction 限制到合法 autoregressive prefix

Multi-Head
  -> 多个 learned interaction subspaces

Residual + Norm + MLP
  -> 可以稳定地堆成深层 repeated Transformer Blocks

Original Transformer
  -> encoder–decoder attention architecture

Decoder-Only GPT
  -> 将 context 与 continuation 统一成 one causal token stream

MiniGPT
  -> embedding + causal Blocks × N + final norm + LM head
```

这就是本项目“from scratch”的真正含义：最终代码不只是会写，还能说明每一行为什么在这里。


## 13. Step 15.5 材料边界

- 完整 forward shape chain 已追踪：`(B,T) -> embeddings (B,T,C) -> every Block internals -> final norm (B,T,C) -> logits (B,T,V)`。
- 当前精确参数公式为 `P = 2VC + T_max*C + N(12C^2+7C) + C = 12NC^2 + C(2V+T_max+7N+1)`。
- 示例 `V=32,C=16,H=4,N=2,T_max=16` 的手算参数总数为 `7664`；H 在当前 fixed-C full-width QKV 设计中不改变参数总数，只改变 `D=C/H` 与 tensor organization。
- token embedding 与 LM head 未 weight-tie，因此两个 `VC` 都独立计入；position table 按完整 `T_max*C` 计入，而不是当前 batch 的 `T*C`。
- causal mask、position IDs、Python metadata 均不计入 trainable parameter total。
- 准备了 MLX parameter-tree count code 与逐步 forward trace helper，但本轮均未运行，不能声称 actual count assertion 或 shape trace runtime passed。
- 已把 token embedding、position representation、causal attention、multi-head、residual、RMSNorm、MLP、Block stacking、final norm、LM head 重新连接到各自历史目的。
- 本轮只准备 Step 15.5；不进入 Step 16，不创建 Tiny Shakespeare 数据管线，不下载数据，不开始 tokenizer / dataset batching / training workflow。
